## Loading in python packages 

In [ ]:
import pandas as pd 
import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob
import os
import re
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature

### Path for data and basic code for air temperature in San Diego 

In [ ]:
CSV = '/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/sshamsian/Forecast_verification_sshamsian/KSAN.2026-05-30.csv'
df = pd.read_csv(
    CSV,
    comment="#",            # drop the metadata block at the top
    skiprows=[11],          # drop the units row
    parse_dates=["Date_Time"],
)
 
df = df.rename(columns={"air_temp_set_1": "temp_F"})
df = df.dropna(subset=["Date_Time", "temp_F"])
df["temp_F"] = pd.to_numeric(df["temp_F"], errors="coerce")
df = df.dropna(subset=["temp_F"]).sort_values("Date_Time")
 
# Average the 5-minute readings into hourly means.
hourly = df.set_index("Date_Time")["temp_F"].resample("h").mean().dropna()
 
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(hourly.index, hourly.values, lw=1.0, color="#3b7dd8",
        label="Hourly mean")
 
ax.set_title("Air Temperature — San Diego International Airport (KSAN)\n"
             "May 2026, hourly averages",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (\u00b0F)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
ax.set_xlim(hourly.index.min(), hourly.index.max())
 
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.autofmt_xdate()
 
fig.tight_layout()
 
print(f"5-min rows: {len(df)}  ->  hourly points: {len(hourly)}")
print(f"Min/Mean/Max hourly temp (F): {hourly.min():.1f} / "
      f"{hourly.mean():.1f} / {hourly.max():.1f}")
 

## Plotting an object 1998 February 1st 24 hour lead time from the output generated by MODE 

In [ ]:
data_path = "/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/Raw_output/1998/1998020100/500/mode_WestWRF_240000L_19980201_000000V_000000A_obj.nc"
ds = xr.open_dataset(data_path)
plt.figure(figsize=(10, 7))
ds["fcst_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Forecast Raw Field February 1, 1998 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

plt.figure(figsize=(10, 7))
ds["obs_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Observed Raw Field February 1, 1998 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

plt.figure(figsize=(10, 7))

ds["fcst_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})

plt.plot(
    ds["fcst_simp_bdy_lon"],
    ds["fcst_simp_bdy_lat"],
    color="green",
    linewidth=2,
    label="Forecast object boundary"
)

plt.plot(
    ds["obs_simp_bdy_lon"],
    ds["obs_simp_bdy_lat"],
    color="black",
    linewidth=2,
    label="Observed object boundary"
)

plt.title("Forecast Raw Field with Forecast and Observed MODE Boundaries February 1, 1998 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

## AR lead times overlaid on one observed data plot on one plot 

In [ ]:

lead_hours = [24, 48, 72, 96, 120, 144, 168]

def parse_lead_hours(path):
    """Extract the forecast lead time in hours from a MODE filename.

    Parameters
    ----------
    path : str
        Path to a MODE netCDF file.

    Returns
    -------
    int or None
        Lead time in hours, or None if no match is found.
    """
    match = re.search(r'_(\d+)L_', os.path.basename(path))
    if match:
        return int(match.group(1)) // 10000
    return None
# globs the netcdf files in the directory and sorts them into a list of paths
nc_files = sorted(glob.glob(
    '/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/Raw_output/1998/1998020100/500/*.nc',
    recursive=True,
))

lead_files = []
for path in nc_files:
    lead = parse_lead_hours(path)
    if lead in lead_hours:
        lead_files.append((lead, path))
lead_files.sort(key=lambda item: item[0])


In [ ]:


lead_hours = [24, 48, 72, 96, 120, 144, 168] # list contianing the lead hours for the ARs 
ivt_level  = 500          # MODE convolution threshold (only used for the field background)
SHOW_FIELD = True         # True = viridis obs IVT field behind the objects, like your figure

# --- helpers functions ----------------
def _bdy_vars(ds, prefix):
    """Resolve the boundary index variable names for a MODE object dataset.

    Parameters
    ----------
    ds : xarray.Dataset
        Opened MODE netCDF dataset.
    prefix : str
        Variable prefix for forecast or observed fields, such as "fcst" or "obs".

    Returns
    -------
    tuple of str
        Names of the start-index and point-count variables for the simplified boundary.

    Raises
    ------
    KeyError
        If the expected boundary variables are not present in the dataset.
    """

    start = f"{prefix}_simp_bdy_start"
    npts  = next((f"{prefix}_simp_bdy_{s}" for s in ("npts", "lens", "count")
                  if f"{prefix}_simp_bdy_{s}" in ds), None)
    if start not in ds or npts is None:
        raise KeyError(f"Couldn't find {prefix} boundary index vars. Available: "
                       f"{[v for v in ds.variables if 'simp_bdy' in v]}")
    return start, npts

def iter_objects(ds, prefix):
    """Yield closed polygon coordinates for each simplified MODE object.

    Parameters
    ----------
    ds : xarray.Dataset
        Opened MODE netCDF dataset.
    prefix : str
        Variable prefix for forecast or observed fields, such as "fcst" or "obs".

    Yields
    ------
    tuple of numpy.ndarray
        Longitude and latitude arrays for one closed object boundary.
    """
    lon = np.asarray(ds[f"{prefix}_simp_bdy_lon"])
    lat = np.asarray(ds[f"{prefix}_simp_bdy_lat"])
    s_name, n_name = _bdy_vars(ds, prefix)
    start = np.asarray(ds[s_name]).astype(int)
    npts  = np.asarray(ds[n_name]).astype(int)
    for s, n in zip(start, npts): # loops through the objects one at a time
        xs, ys = lon[s:s+n], lat[s:s+n]
        yield np.append(xs, xs[0]), np.append(ys, ys[0]) # the yield hands back that objects closed [lat,lon] polygon. The yield makes this functions as a generator, instead of building a list of all objects up front 

# --------------------------------------------------------------------------
# This walks over my nc_files list and keeps only the ones whose parsed lead is in by original list in the previous code cell 
# and then it pairs each path with its lead as (lead, path).sorted then order them by lead ascending (basically tuples sort by their first element), so that the plot and legend come out 24-68 in order 
lead_files = sorted(
    (parse_lead_hours(p), p) for p in nc_files
    if parse_lead_hours(p) in lead_hours
)
# this is just in case the lead_files have nothing in them and are completely empty basically just a safe guard 
if not lead_files:
    print('No forecast files found for lead times 24-168 h. '
          'Update the path or extract the .nc files first.')
else:
    print('Found lead times:', [lead for lead, _ in lead_files])
    proj = ccrs.LambertConformal(
        central_longitude=-140,
        central_latitude=45,
        standard_parallels=(30, 60))
    
    fig, ax = plt.subplots(figsize=(9, 7),subplot_kw={'projection': proj})
    # Set the map extent to cover the AK -> CA West Coast domain
    # [lon_min, lon_max, lat_min, lat_max]
    ax.set_extent([-170, -110, 25, 62], crs=ccrs.PlateCarree())

    # --- background features ---
    ax.add_feature(cfeature.OCEAN, facecolor='#cfe3ee', zorder=0)
    ax.add_feature(cfeature.LAND, facecolor='#f0efe9', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=1)

    # State boundaries (includes Alaska + California)
    states = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='gray', linewidth = 0.6, zorder=1)


    # Observed: same valid time across all leads -> load once, draw once.
    # this basically open the first file in the lead_files list and then it extracts the observed field from that file and then it plots it as a background field behind the forecast objects
    ds0 = xr.open_dataset(lead_files[0][1])
    obs = ds0['obs_raw']

    if SHOW_FIELD:                              # plasma IVT field behind everything
        obs.plot(ax=ax, x='lon', y='lat', cmap='plasma_r', zorder=0,transform = ccrs.PlateCarree(),
                 cbar_kwargs={'label': 'Observed IVT (kg m$^{-1}$ s$^{-1}$)'}, vmin = 250, vmax = 1000)
    # this basically open the first file in the lead_files list and then it extracts the observed field from that file and then it plots it as a background field behind the forecast objects    plt.show()


    ds0 = xr.open_dataset(lead_files[0][1])    
    plt.tight_layout()

    for xs, ys in iter_objects(ds0, 'obs'):    # shade + outline the observed object and loop through the observed objects 
        ax.fill(xs, ys, facecolor='black', alpha=0.30, zorder=3,transform = ccrs.PlateCarree()) # alpha =0.3 makes it 30 percent opaque so the IVT field still shows through
        ax.plot(xs, ys, color='black', lw=3.5, zorder=4, transform=ccrs.PlateCarree()) # ax.plot draw the solid black outline on top, and the z order numbers what is drawn first and on top of what.

    # Forecast: one boundary per lead, colored by lead time.
    colors  = plt.cm.coolwarm(np.linspace(0.0, 0.85, len(lead_files))) # samples N evenly spaced colors along the plasma colored map one per lead, stopping at 0.85 to avoid the pale yellow tail 
    handles = [Line2D([0], [0], color='black', lw=2.5, label='Observed')] # this is just a manual legend Each forecast lead draws potentially several line segments, and letting matplotlib auto-build the legend from those would give me a cluttered, repeated mess.
    # 
    for (lead, path), color in zip(lead_files, colors):
        ds = xr.open_dataset(path)
        if 'fcst_simp_bdy_lon' not in ds:
            continue
        for i, (xs, ys) in enumerate(iter_objects(ds, 'fcst')):
            ax.plot(xs, ys, color=color, lw=1.8, zorder=5, transform=ccrs.PlateCarree())
        handles.append(Line2D([0], [0], color=color, lw=1.8, label=f'{lead} h'))

    ax.legend(handles=handles, title='Forecast lead', loc='upper right', framealpha=0.9)
    ax.set_title('AR object position by forecast lead — valid 1998-02-01 00Z')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()